# Подбор гиперпараметров при помощи Optuna. Практика

## Импорты, переменные окружения и константы

In [7]:
import os
from collections import defaultdict
import numpy as np
import pandas as pd
import psycopg
import optuna
from optuna.samplers import TPESampler
from optuna.integration.mlflow import MLflowCallback
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
)
from catboost import CatBoostClassifier
import mlflow
from mlflow.utils.mlflow_tags import MLFLOW_PARENT_RUN_ID
from dotenv import load_dotenv
from numpy import array, median

# Загружаем креды к БД и S3 из .env
load_dotenv()

# Настраиваем подключение к вашему S3-бакету в Yandex Cloud (как в прошлом ноутбуке)
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ["S3_BUCKET_NAME"] = os.getenv("S3_BUCKET_NAME")
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

EXPERIMENT_NAME = "Hyperparameter_tuning_with_Optuna_lesson_6"
RUN_NAME = "model_bayesian_search"
REGISTRY_MODEL_NAME = "hyperparameter_tuning_model"

TABLE_NAME = "users_churn"
STUDY_DB_NAME = "sqlite:///local.study.db"
STUDY_NAME = "churn_model"

## Загрузка данных из вашей БД и разбиение выборки

In [2]:
connection = {"sslmode": "require", "target_session_attrs": "read-write"}
postgres_credentials = {
    "host": os.getenv("DB_DESTINATION_HOST"),
    "port": os.getenv("DB_DESTINATION_PORT"),
    "dbname": os.getenv("DB_DESTINATION_NAME"),
    "user": os.getenv("DB_DESTINATION_USER"),
    "password": os.getenv("DB_DESTINATION_PASSWORD"),
    "sslmode": "disable",
}
connection.update(postgres_credentials)

with psycopg.connect(**connection) as conn:
    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM destination.{TABLE_NAME}")
        data = cur.fetchall()
        columns = [col.name for col in cur.description]

df = pd.DataFrame(data, columns=columns)

# Очистка пропусков и разбиение выборки
df["total_charges"] = df["total_charges"].fillna(0)

features = ["monthly_charges", "total_charges", "senior_citizen"]
target = "target"
split_column = "begin_date"
test_size = 0.2

df = df.sort_values(by=[split_column])

X_train, X_test, y_train, y_test = train_test_split(
    df[features],
    df[target],
    test_size=test_size,
    shuffle=False,
)

print(f"Размер выборки для обучения: {X_train.shape}")
print(f"Размер выборки для теста: {X_test.shape}")

Размер выборки для обучения: (5634, 3)
Размер выборки для теста: (1409, 3)


## Целевая функция оптимизации objective

In [3]:
from numpy import array, median


def objective(trial: optuna.Trial) -> float:
    param = {
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.1, log=True),
        "depth": trial.suggest_int("depth", 1, 12),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 0.1, 5),
        "random_strength": trial.suggest_float("random_strength", 0.1, 5),
        "loss_function": "Logloss",
        "task_type": "CPU",
        "random_seed": 0,
        "iterations": 300,
        "verbose": False,
    }
    model = CatBoostClassifier(**param)

    skf = StratifiedKFold(n_splits=2)

    metrics = defaultdict(list)
    for i, (train_index, val_index) in enumerate(skf.split(X_train, y_train)):
        train_x = X_train.iloc[train_index]
        val_x = X_train.iloc[val_index]
        train_y = y_train.iloc[train_index]
        val_y = y_train.iloc[val_index]

        model.fit(train_x, train_y)
        prediction = model.predict(val_x)
        probas = model.predict_proba(val_x)[:, 1]

        _, err1, _, err2 = confusion_matrix(val_y, prediction, normalize="all").ravel()
        auc = roc_auc_score(val_y, probas)
        precision = precision_score(val_y, prediction)
        recall = recall_score(val_y, prediction)
        f1 = f1_score(val_y, prediction)
        logloss = log_loss(val_y, prediction)

        metrics["err1"].append(err1)
        metrics["err2"].append(err2)
        metrics["auc"].append(auc)
        metrics["precision"].append(precision)
        metrics["recall"].append(recall)
        metrics["f1"].append(f1)
        metrics["logloss"].append(logloss)

    err_1 = median(array(metrics["err1"]))
    err_2 = median(array(metrics["err2"]))
    auc = median(array(metrics["auc"]))
    precision = median(array(metrics["precision"]))
    recall = median(array(metrics["recall"]))
    f1 = median(array(metrics["f1"]))
    logloss = median(array(metrics["logloss"]))

    return auc

## Запуск подбора гиперпараметров (Optuna)

In [4]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if not experiment:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

mlflc = MLflowCallback(
    tracking_uri=f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}",
    metric_name="AUC",
    create_experiment=False,
    mlflow_kwargs={"experiment_id": experiment_id, "tags": {MLFLOW_PARENT_RUN_ID: run_id}},
)

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STUDY_DB_NAME,
    sampler=optuna.samplers.TPESampler(),
    direction="maximize",
    load_if_exists=True,
)
study.optimize(objective, n_trials=10, callbacks=[mlflc])
best_params = study.best_params

print(f"Number of finished trials: {len(study.trials)}")
print(f"Best params: {best_params}")


/tmp/ipykernel_1516/2295300971.py:10: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflc = MLflowCallback(
[I 2026-11-23 04:05:31,990] A new study created in RDB with name: churn_model
/home/mle-user/projects/mle-mlflow/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
[I 2026-11-23 04:05:34,054] Trial 0 finished with value: 0.7156897336849477 and parameters: {'learning_rate': 0.005177790340361488, 'depth': 10, 'l2_leaf_reg': 1.387101515795677, 'random_strength': 4.204292503997319}. Best is trial 0 with value: 0.7156897336849477.
/home/mle-user/projects/mle-mlflow/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: P

Number of finished trials: 10
Best params: {'learning_rate': 0.06770131998120088, 'depth': 3, 'l2_leaf_reg': 1.042651946620327, 'random_strength': 0.9339234741368082}


## Обучение и логирование лучшей модели в MLflow

In [8]:
best_model = CatBoostClassifier(
    **best_params,
    loss_function="Logloss",
    task_type="CPU",
    random_seed=0,
    iterations=300,
    verbose=False,
)
best_model.fit(X_train, y_train)

prediction_test = best_model.predict(X_test)
probas_test = best_model.predict_proba(X_test)[:, 1]
_, err1_test, _, err2_test = confusion_matrix(y_test, prediction_test, normalize="all").ravel()

test_metrics = {
    "err1": err1_test,
    "err2": err2_test,
    "auc": roc_auc_score(y_test, probas_test),
    "precision": precision_score(y_test, prediction_test),
    "recall": recall_score(y_test, prediction_test),
    "f1": f1_score(y_test, prediction_test),
    "logloss": log_loss(y_test, prediction_test),
}

signature = mlflow.models.infer_signature(X_test, prediction_test)
input_example = X_test[:10]

with mlflow.start_run(run_id=run_id):
    mlflow.log_params(best_params)
    mlflow.log_metrics(test_metrics)
    mlflow.catboost.log_model(
        cb_model=best_model,
        artifact_path="models",
        registered_model_name=REGISTRY_MODEL_NAME,
        signature=signature,
        input_example=input_example,
        pip_requirements=["catboost", "scikit-learn", "pandas", "optuna==3.4.0"],
    )

/home/mle-user/projects/mle-mlflow/.venv/lib/python3.10/site-packages/mlflow/models/signature.py:212: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  inputs = _infer_schema(model_input) if model_input is not None else None
Registered model 'hyperparameter_tuning_model' already exists. Creating a new version of this model...
2026/11/23 04:08:42 INFO mlflow.tracking._model_regi